# TTS bootcamp – run 2: Task 3 (idea) + Task 4 (faster)
Settings: **GPU T4 x2**, Internet On, secret HF_TOKEN on, input `tts-bootcamp`. Save & Run All.

In [ ]:
!nvidia-smi
import sys, platform; print(sys.version)

In [ ]:
import os, shutil, subprocess
src = None
for root, dirs, files in os.walk('/kaggle/input'):
    if 'metrics.py' in files and 'baseline' in dirs:
        src = root; break
assert src, 'kit folder (with metrics.py) not found under /kaggle/input'
W = '/kaggle/working/tts'
shutil.rmtree(W, ignore_errors=True)
shutil.copytree(src, W)
os.chdir(W)
print('kit at', src, '->', W, sorted(os.listdir(W)))

def sh(cmd, tail=60):
    print('\n$', cmd, flush=True)
    env = dict(os.environ, PYTHONPATH=os.path.join(W, 'code', 'compat'))
    r = subprocess.run(cmd, shell=True, text=True, capture_output=True, env=env)
    out = (r.stdout + ('\n' + r.stderr if r.returncode else '')).strip().splitlines()
    print('\n'.join(out[-tail:]))
    print('[exit code]', r.returncode, flush=True)
    return r.returncode

In [ ]:
# compatibility shim (also shipped in code/compat/): soundfile fallback for torchaudio.load
os.makedirs('code/compat', exist_ok=True)
open('code/compat/sitecustomize.py', 'w').write('"""Compatibility shim (loaded automatically when code/compat is on PYTHONPATH).\n\ntorchaudio >= 2.9 reads audio only through the optional `torchcodec` package. If that path fails,\nfall back to soundfile so IndicF5\'s reference-clip loading still works. Audio content is unchanged.\n"""\ntry:\n    import torchaudio as _ta\n\n    _orig_load = _ta.load\n\n    def _load(path, *args, **kwargs):\n        try:\n            return _orig_load(path, *args, **kwargs)\n        except Exception:\n            import soundfile as _sf\n            import torch as _torch\n            y, sr = _sf.read(str(path), dtype="float32", always_2d=True)\n            return _torch.from_numpy(y.T.copy()), sr\n\n    _ta.load = _load\nexcept Exception:\n    pass\n')
print('shim written')

In [ ]:
from kaggle_secrets import UserSecretsClient
os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')
from huggingface_hub import login
login(token=os.environ['HF_TOKEN'])
print('HF login ok')

In [ ]:
# NOTE: the kit's requirements.txt pins IndicF5 to fb652a9..., which is the Hugging Face *model* revision,
# not a commit of the GitHub code repo, so `pip install -r baseline/requirements.txt` fails (git exit 128).
# We install the same dependencies explicitly and the IndicF5 code at its latest GitHub commit (13f7c4d).
# audioop-lts: pydub needs it on Python 3.13. torchcodec: torchaudio>=2.9 needs it to read wav files.
import torch
keep = f'"torch=={torch.__version__}"'
try:
    import torchaudio; keep += f' "torchaudio=={torchaudio.__version__}"'
except ImportError:
    keep += f' "torchaudio=={torch.__version__.split('+')[0]}"'
pkgs = ('"numpy>=2.1,<2.3" "transformers==4.49.0" accelerate soundfile librosa pyyaml pandas "speechbrain>=1.0" '
        'cached_path ema_pytorch hydra-core jieba pypinyin pydub torchdiffeq vocos "x_transformers>=1.31.14" '
        'transformers_stream_generator safetensors tomli click wandb datasets audioop-lts torchcodec')
sh(f'pip install -q {keep} {pkgs}', tail=15)
sh('pip install -q --no-deps "git+https://github.com/ai4bharat/IndicF5.git@13f7c4d627cc10111aea8fe9c0039462cacacdc7"', tail=15)
sh('python -c "import f5_tts, torchaudio, numpy, transformers, torch, speechbrain; print(numpy.__version__, transformers.__version__, torch.__version__, torchaudio.__version__)"')

In [ ]:
sh('python code/warmup.py')
sh('python code/warmup.py')

In [ ]:
# write this run's code files (identical to code/ in the submission)
os.makedirs('code', exist_ok=True)
open('code/normalize.py', 'w', encoding='utf-8').write('"""\nTask 2 - rule-based text frontend for IndicF5 (no model of any kind).\n\nnormalize(text) rewrites everything that is not already a speakable word into Hindi words in Devanagari:\n  * Indian-style cardinal numbers (हज़ार, लाख, करोड़), Devanagari digits included\n  * ₹ amounts (with लाख/करोड़ units and paise), percentages, decimals\n  * dates dd/mm/yyyy and "<day> <month> <year>", years 1100-1999 read as "उन्नीस सौ ..."\n  * abbreviations written in capitals (RBI, ATM, GST ...) spelled out letter by letter\nEnglish words written in Latin script are left unchanged (that is a Task 3 problem).\n\nAll rules were written from the validation items (and standard Hindi number names); nothing was\nderived from the evaluation text.\n\n    python code/normalize.py --kit . --out predictions/t2_normalized.csv\n"""\nimport argparse\nimport csv\nimport re\nfrom pathlib import Path\n\nONES = ("शून्य एक दो तीन चार पाँच छह सात आठ नौ दस ग्यारह बारह तेरह चौदह पंद्रह सोलह सत्रह अठारह उन्नीस बीस "\n        "इक्कीस बाईस तेईस चौबीस पच्चीस छब्बीस सत्ताईस अट्ठाईस उनतीस तीस इकतीस बत्तीस तैंतीस चौंतीस पैंतीस "\n        "छत्तीस सैंतीस अड़तीस उनतालीस चालीस इकतालीस बयालीस तैंतालीस चवालीस पैंतालीस छियालीस सैंतालीस अड़तालीस "\n        "उनचास पचास इक्यावन बावन तिरेपन चौवन पचपन छप्पन सत्तावन अट्ठावन उनसठ साठ इकसठ बासठ तिरेसठ चौंसठ "\n        "पैंसठ छियासठ सड़सठ अड़सठ उनहत्तर सत्तर इकहत्तर बहत्तर तिहत्तर चौहत्तर पचहत्तर छिहत्तर सतहत्तर अठहत्तर "\n        "उन्यासी अस्सी इक्यासी बयासी तिरासी चौरासी पचासी छियासी सत्तासी अट्ठासी नवासी नब्बे इक्यानवे बानवे "\n        "तिरानवे चौरानवे पचानवे छियानवे सत्तानवे अट्ठानवे निन्यानवे").split()\nassert len(ONES) == 100\n\nMONTHS = ["जनवरी", "फ़रवरी", "मार्च", "अप्रैल", "मई", "जून", "जुलाई", "अगस्त", "सितंबर", "अक्टूबर", "नवंबर", "दिसंबर"]\n\n# English letter names as said in Indian English (RBI -> आर बी आई, ATM -> ए टी एम, GST -> जी एस टी)\nLETTERS = {"A": "ए", "B": "बी", "C": "सी", "D": "डी", "E": "ई", "F": "एफ़", "G": "जी", "H": "एच", "I": "आई",\n           "J": "जे", "K": "के", "L": "एल", "M": "एम", "N": "एन", "O": "ओ", "P": "पी", "Q": "क्यू", "R": "आर",\n           "S": "एस", "T": "टी", "U": "यू", "V": "वी", "W": "डब्ल्यू", "X": "एक्स", "Y": "वाई", "Z": "ज़ेड"}\n\nDEV_DIGITS = str.maketrans("०१२३४५६७८९", "0123456789")\nUNITS = "लाख|करोड़|करोड़|हज़ार|हजार"\n\n\ndef cardinal(n: int) -> str:\n    """Indian-system number words: 1,47,382 -> एक लाख सैंतालीस हज़ार तीन सौ बयासी."""\n    if n < 100:\n        return ONES[n]\n    parts = []\n    for value, word in ((10 ** 7, "करोड़"), (10 ** 5, "लाख"), (1000, "हज़ार"), (100, "सौ")):\n        if n >= value:\n            q, n = divmod(n, value)\n            parts.append(f"{cardinal(q)} {word}")\n    if n:\n        parts.append(ONES[n])\n    return " ".join(parts)\n\n\ndef year_words(n: int) -> str:\n    """Years 1100-1999 are read in pairs (उन्नीस सौ निन्यानवे); others as cardinals (दो हज़ार छब्बीस)."""\n    if 1100 <= n <= 1999:\n        hi, lo = divmod(n, 100)\n        return f"{ONES[hi]} सौ" + (f" {ONES[lo]}" if lo else "")\n    return cardinal(n)\n\n\ndef digits_words(s: str) -> str:\n    return " ".join(ONES[int(c)] for c in s)\n\n\ndef number_words(s: str) -> str:\n    """\'1,25,000\' / \'12951\' / \'2.3\' -> words. Digits after a decimal point are read one by one."""\n    s = s.replace(",", "")\n    if "." in s:\n        whole, frac = s.split(".", 1)\n        return f"{cardinal(int(whole or 0))} दशमलव {digits_words(frac)}"\n    return cardinal(int(s))\n\n\nNUM = r"\\d[\\d,]*(?:\\.\\d+)?"\n\n\ndef _money(m):\n    amount, unit = m.group(1), m.group(2)\n    amount = amount.rstrip(",.")\n    if unit:                                   # ₹38 लाख -> अड़तीस लाख रुपये\n        return f"{number_words(amount)} {unit} रुपये"\n    clean = amount.replace(",", "")\n    if "." in clean:                           # ₹2,340.50 -> ... रुपये पचास पैसे\n        rupees, paise = clean.split(".", 1)\n        paise = (paise + "0")[:2]\n        out = f"{cardinal(int(rupees or 0))} रुपये"\n        return out + (f" {cardinal(int(paise))} पैसे" if int(paise) else "")\n    return f"{cardinal(int(clean))} रुपये"\n\n\ndef _date(m):\n    d, mo, y = int(m.group(1)), int(m.group(2)), m.group(3)\n    if not (1 <= d <= 31 and 1 <= mo <= 12):\n        return m.group(0)\n    year = int(y) if len(y) == 4 else 2000 + int(y)\n    return f"{cardinal(d)} {MONTHS[mo - 1]} {year_words(year)}"\n\n\ndef _plain_number(m):\n    s = m.group(0)\n    if "," not in s and "." not in s and len(s) == 4 and 1100 <= int(s) <= 1999:\n        return year_words(int(s))\n    return number_words(s)\n\n\ndef _abbrev(m):\n    return " ".join(LETTERS[c] for c in m.group(0))\n\n\ndef normalize(text: str) -> str:\n    t = text.translate(DEV_DIGITS)\n    # dates: 12/03/2026, 5-6-2026\n    t = re.sub(r"(?<!\\d)(\\d{1,2})[/\\-.](\\d{1,2})[/\\-.](\\d{4}|\\d{2})(?!\\d)", _date, t)\n    # rupee amounts, optionally followed by a unit word\n    t = re.sub(rf"(?:₹|रु\\.?|Rs\\.?)\\s*({NUM})(?:\\s*({UNITS}))?", _money, t)\n    # percentages\n    t = re.sub(rf"({NUM})\\s*%", lambda m: f"{number_words(m.group(1))} प्रतिशत", t)\n    # a hyphen glued to a number (चंद्रयान-3) is read as a pause, not a minus\n    t = re.sub(r"(?<=[ऀ-ॿ])-(?=\\d)", " ", t)\n    # remaining numbers\n    t = re.sub(r"\\d[\\d,]*\\d(?:\\.\\d+)?|\\d(?:\\.\\d+)?", _plain_number, t)\n    # abbreviations: two or more capitals (SBI, KYC); spelled letter by letter\n    t = re.sub(r"\\b[A-Z]{2,}\\b", _abbrev, t)\n    t = t.replace("₹", " रुपये ")\n    t = re.sub(r"\\s+([,।?!.])", r"\\1", t)\n    return re.sub(r"\\s+", " ", t).strip()\n\n\ndef main():\n    ap = argparse.ArgumentParser()\n    ap.add_argument("--kit", default=".", help="folder that holds data/val_items.csv and data/eval_items.csv")\n    ap.add_argument("--out", default="predictions/t2_normalized.csv")\n    a = ap.parse_args()\n    rows = []\n    for split, name in (("val", "val_items.csv"), ("eval", "eval_items.csv")):\n        with open(Path(a.kit) / "data" / name, newline="", encoding="utf-8") as f:\n            for r in csv.DictReader(f):\n                rows.append([r["id"], split, r["text"], normalize(r["text"])])\n    Path(a.out).parent.mkdir(parents=True, exist_ok=True)\n    with open(a.out, "w", newline="", encoding="utf-8") as f:\n        w = csv.writer(f)\n        w.writerow(["id", "split", "text", "normalized_text"])\n        w.writerows(rows)\n    print(f"wrote {len(rows)} rows to {a.out}")\n\n\nif __name__ == "__main__":\n    main()\n')
open('code/translit.py', 'w', encoding='utf-8').write('"""\nTask 3 idea - rule-based Indian-English spelling -> Devanagari converter for Latin-script words.\n\nNo model and no word list: every English word is converted by general letter-to-sound rules\n(grapheme patterns with left/right context), written from the validation items and general\nIndian-English pronunciation, never from the evaluation text.\n\nPipeline: original text -> Task 2 normalize() -> transliterate_text() -> IndicF5.\n\nRules, in short\n  * digraphs/trigraphs: sh->श, ch->च, th->थ, ph->फ, ck->क, gh (silent after i), kn->न, wr->र,\n    tion/sion->शन, qu->क्व, x->क्स, ng->ंग\n  * vowels: ee/ea->ी, oo->ू, ou/ow(+n)->ाउ, ow(final)->ो, ai/ay->े, ew->यू, igh->ाइ,\n    magic-e (a_e->े, i_e->ाइ, o_e->ो, u_e->यू), silent final e, final y->ी, ie(final)->ी\n  * Indian-English vowel colours: o in a closed syllable -> ॉ (office, shopping),\n    a in a closed syllable -> ै (cancel, match), a before ss/st/sk/r -> ा (class, fast, party),\n    al+k/l -> ॉ (walk, call), word-initial a + doubled consonant -> ए (address)\n  * c/g softening before e/i/y, -ge/-dge -> ज\n  * n/m before a consonant -> anusvara (internet -> इंटरनेट, weekend -> वीकेंड)\n  * doubled consonants are said once; consonant clusters get a virama; a word-final consonant\n    keeps Hindi\'s silent final schwa\n"""\nimport re\n\nV = "V"   # vowel unit: (V, independent, matra)\nC = "C"   # consonant unit: (C, letter)\nN = "N"   # nasal mark (anusvara)\n\nCONS = {"b": "ब", "c": "क", "d": "ड", "f": "फ", "g": "ग", "h": "ह", "j": "ज", "k": "क", "l": "ल",\n        "m": "म", "n": "न", "p": "प", "q": "क", "r": "र", "s": "स", "t": "ट", "v": "व", "w": "व",\n        "x": "क्स", "y": "य", "z": "ज़"}\n\nVOW = {  # name: (independent form, matra)\n    "a": ("अ", ""), "aa": ("आ", "ा"), "i": ("इ", "ि"), "ii": ("ई", "ी"), "u": ("उ", "ु"),\n    "uu": ("ऊ", "ू"), "e": ("ए", "े"), "ai": ("ऐ", "ै"), "o": ("ओ", "ो"), "au": ("औ", "ौ"),\n    "O": ("ऑ", "ॉ"), "aai": ("आइ", "ाइ"), "aau": ("आउ", "ाउ"), "yuu": ("यू", "्यू"),\n}\nVOWELS = set("aeiou")\n\n\ndef _is_cons(ch):\n    return ch.isalpha() and ch not in VOWELS\n\n\ndef _closed(w, j):\n    """True if position j (just after a vowel) starts a closed syllable: two consonants follow, or one\n    consonant then end of word (ignoring a silent final e)."""\n    rest = w[j:]\n    if rest.endswith("e") and len(rest) >= 2 and _is_cons(rest[-2]) and len(rest) == 2:\n        return False  # magic-e handled elsewhere\n    if len(rest) == 0:\n        return False\n    if len(rest) == 1:\n        return _is_cons(rest[0])\n    return _is_cons(rest[0]) and (_is_cons(rest[1]) or rest[1] == "")\n\n\ndef word_units(w):\n    """Convert one lowercase Latin word into a list of units."""\n    u, i, n = [], 0, len(w)\n\n    def magic_e(k):  # vowel at k, then one consonant, then a final \'e\'\n        if not (k + 2 == n - 1 and _is_cons(w[k + 1]) and w[n - 1] == "e" and w[k + 1] not in "wxy"):\n            return False\n        # unstressed endings in longer words keep a short i: office, notice, active, favourite\n        if w[k] == "i" and any(c in VOWELS for c in w[:k]) and w[k + 1] in "cvt":\n            return False\n        return True\n\n    while i < n:\n        s = w[i:]\n        prev = w[i - 1] if i else ""\n        # ---- multi-letter patterns\n        if s.startswith(("tion", "sion")):\n            u += [(C, "श"), (V, "a")]; u += [(C, "न")]; i += 4; continue\n        if s.startswith("igh"):\n            u.append((V, "aai")); i += 3; continue\n        if s.startswith("dge"):\n            u.append((C, "ज")); i += 3; continue\n        if s.startswith("ck"):\n            u.append((C, "क")); i += 2; continue\n        if s.startswith("sh"):\n            u.append((C, "श")); i += 2; continue\n        if s.startswith("ch"):\n            u.append((C, "च")); i += 2; continue\n        if s.startswith("th"):\n            u.append((C, "थ")); i += 2; continue\n        if s.startswith("ph"):\n            u.append((C, "फ")); i += 2; continue\n        if s.startswith("wh"):\n            u.append((C, "व")); i += 2; continue\n        if s.startswith("qu"):\n            u += [(C, "क"), (C, "व")]; i += 2; continue\n        if i == 0 and s.startswith("kn"):\n            u.append((C, "न")); i += 2; continue\n        if i == 0 and s.startswith("wr"):\n            u.append((C, "र")); i += 2; continue\n        if s.startswith("ng"):\n            u += [(N, ""), (C, "ग")]; i += 2; continue\n        if s.startswith(("ee", "ea")):\n            u.append((V, "ii")); i += 2; continue\n        if s.startswith("oo"):\n            u.append((V, "uu")); i += 2; continue\n        if s.startswith("ou"):\n            u.append((V, "aau")); i += 2; continue\n        if s.startswith("ow"):\n            u.append((V, "o" if i + 2 >= n else "aau")); i += 2; continue\n        if s.startswith(("ai", "ay")):\n            u.append((V, "e")); i += 2; continue\n        if s.startswith("ew"):\n            u.append((V, "yuu")); i += 2; continue\n        if s.startswith("oa"):\n            u.append((V, "o")); i += 2; continue\n        if s.startswith("ie") and i + 2 == n:\n            u.append((V, "ii")); i += 2; continue\n        if s.startswith("al") and i + 2 < n and w[i + 2] in "kl":\n            u += [(V, "O")]; i += 1 + (w[i + 2] == "k"); continue   # walk -> वॉक, call -> कॉल\n        # ---- single letters\n        ch = w[i]\n        if ch == "e":\n            if i == n - 1 and n > 2:\n                i += 1; continue                       # silent final e\n            if s.startswith("er") and i + 2 == n:\n                u += [(V, "a"), (C, "र")]; i += 2; continue            # proper -> प्रॉपर\n            if s.startswith("er") and i + 2 < n and _is_cons(w[i + 2]):\n                u += [(V, "a"), (C, "र"), (V, "a")]; i += 2; continue  # internet -> इंटरनेट\n            if s.startswith("el") and i + 2 == n:\n                u.append((V, "a")); i += 1; continue   # hotel -> होटल\n            u.append((V, "e")); i += 1; continue\n        if ch == "a":\n            if i == n - 1:\n                u.append((V, "aa"))\n            elif magic_e(i):\n                u.append((V, "e"))\n            elif s.startswith(("ar", "ss", "st", "sk", "sp")) or w[i + 1:i + 3] in ("ss", "st", "sk", "sp"):\n                u.append((V, "aa"))\n            elif i == 0 and i + 2 < n and w[i + 1] == w[i + 2]:\n                u.append((V, "e"))\n            elif _is_cons(w[i + 1]):\n                u.append((V, "ai"))\n            else:\n                u.append((V, "a"))\n            i += 1; continue\n        if ch == "i":\n            u.append((V, "aai" if magic_e(i) else "i")); i += 1; continue\n        if ch == "o":\n            if i == n - 1 or magic_e(i):\n                u.append((V, "o"))\n            elif i + 1 < n and w[i + 1] == "v" and i + 2 < n and w[i + 2] in "ie":\n                u.append((V, "uu"))                    # movie, move\n            elif i + 1 < n and _is_cons(w[i + 1]):\n                u.append((V, "O"))                     # office, shopping, online\n            else:\n                u.append((V, "o"))\n            i += 1; continue\n        if ch == "u":\n            if magic_e(i) or (i + 2 == n and w[i + 1] == "e"):\n                u.append((V, "yuu"))\n            elif s.startswith("ur"):\n                u.append((V, "a"))\n            elif i + 1 < n and _is_cons(w[i + 1]) and i + 2 < n and _is_cons(w[i + 2]):\n                u.append((V, "a"))                     # function, number\n            else:\n                u.append((V, "u"))\n            i += 1; continue\n        if ch == "y":\n            if i == 0:\n                u.append((C, "य"))\n            elif i == n - 1:\n                u.append((V, "ii"))\n            else:\n                u.append((V, "aai"))\n            i += 1; continue\n        if ch == "c":\n            nxt = w[i + 1] if i + 1 < n else ""\n            u.append((C, "स" if nxt in ("e", "i", "y") else "क")); i += 1; continue\n        if ch == "g":\n            nxt = w[i + 1] if i + 1 < n else ""\n            soft = nxt == "e" and i + 2 == n                # message -> मैसेज\n            u.append((C, "ज" if soft else "ग")); i += 1; continue\n        if ch in "nm" and i + 1 < n and w[i + 1] in "kgcjtdpbsfqxz" and u and u[-1][0] == V:\n            u.append((N, "")); i += 1; continue\n        if ch == "h" and prev and _is_cons(prev):\n            i += 1; continue\n        if ch in CONS:\n            if u and u[-1][0] == C and w[i - 1:i] == ch:   # doubled consonant said once\n                i += 1; continue\n            u.append((C, CONS[ch])); i += 1; continue\n        i += 1\n    return u\n\n\ndef render(units):\n    out, prev = [], None\n    for k, (kind, val) in enumerate(units):\n        if kind == C:\n            if prev == C:\n                out.append("्")\n            out.append(val)\n        elif kind == V:\n            ind, mat = VOW[val]\n            out.append(mat if prev == C else ind)\n        else:\n            out.append("ं")\n        prev = kind if kind != N else prev\n    return "".join(out)\n\n\ndef transliterate_word(word):\n    lower = word.lower()\n    if not lower.isalpha():\n        return word\n    return render(word_units(lower))\n\n\ndef transliterate_text(text):\n    """Replace every Latin-script word with its Devanagari reading; everything else is unchanged."""\n    return re.sub(r"[A-Za-z]+", lambda m: transliterate_word(m.group(0)), text)\n\n\nif __name__ == "__main__":\n    import sys\n    for w in sys.argv[1:]:\n        print(w, "->", transliterate_word(w))\n')
open('code/make_t3_text.py', 'w', encoding='utf-8').write('"""\nTask 3 text: original text -> Task 2 normalize() -> rule-based English->Devanagari (translit.py).\n\n    python code/make_t3_text.py --kit . --out out/t3_text.csv\nWrites id,split,text,t3_text for all 64 items (only Latin-script words differ from the Task 2 text).\n"""\nimport argparse\nimport csv\nimport sys\nfrom pathlib import Path\n\nsys.path.insert(0, str(Path(__file__).resolve().parent))\nfrom normalize import normalize          # noqa: E402\nfrom translit import transliterate_text  # noqa: E402\n\n\ndef main():\n    ap = argparse.ArgumentParser()\n    ap.add_argument("--kit", default=".")\n    ap.add_argument("--out", default="out/t3_text.csv")\n    a = ap.parse_args()\n    rows = []\n    for split, name in (("val", "val_items.csv"), ("eval", "eval_items.csv")):\n        with open(Path(a.kit) / "data" / name, newline="", encoding="utf-8") as f:\n            for r in csv.DictReader(f):\n                rows.append([r["id"], split, r["text"], transliterate_text(normalize(r["text"]))])\n    Path(a.out).parent.mkdir(parents=True, exist_ok=True)\n    with open(a.out, "w", newline="", encoding="utf-8") as f:\n        w = csv.writer(f)\n        w.writerow(["id", "split", "text", "t3_text"])\n        w.writerows(rows)\n    print(f"wrote {len(rows)} rows to {a.out}")\n\n\nif __name__ == "__main__":\n    main()\n')
print('code written')

In [ ]:
import subprocess, time
def run_bg(cmd, gpu, log):
    env = dict(os.environ, PYTHONPATH=os.path.join(W, 'code', 'compat'), CUDA_VISIBLE_DEVICES=str(gpu))
    return subprocess.Popen(cmd, shell=True, env=env, stdout=open(log, 'w'), stderr=subprocess.STDOUT)
def tail(log, n=6):
    print(f'--- {log}'); print(''.join(open(log).readlines()[-n:]))
os.makedirs('out', exist_ok=True)
sh('python code/make_t3_text.py --kit . --out out/t3_text.csv')
T3 = '--text-csv out/t3_text.csv --text-column t3_text --seed 0'
# GPU 0: the timed pair for Task 4 (baseline, then the fast system) ; GPU 1: Task 3 validation + evaluation audio
g0 = run_bg(f'python baseline/run_baseline.py --items data/val_items.csv --out out/baseline_val --seed 0 && '
            f'python baseline/run_baseline.py --items data/val_items.csv --out out/val_fast {T3} --set nfe_step=16', 0, 'out/log_gpu0.txt')
g1 = run_bg(f'python baseline/run_baseline.py --items data/val_items.csv --out out/val_idea {T3} && '
            f'python baseline/run_baseline.py --items data/eval_items.csv --out out/t3_idea {T3}', 1, 'out/log_gpu1.txt')
t0 = time.time()
while g0.poll() is None or g1.poll() is None:
    time.sleep(60); print(f'{(time.time()-t0)/60:.0f} min ...', flush=True)
print('exit codes', g0.returncode, g1.returncode)
tail('out/log_gpu0.txt'); tail('out/log_gpu1.txt')

In [ ]:
sh('python metrics.py out/val_idea --json out/val_idea.json --csv out/val_idea.csv', tail=40)
sh('python metrics.py out/val_fast --json out/val_fast.json --csv out/val_fast.csv', tail=12)
import json
idea = json.load(open('out/val_idea.json')); fast = json.load(open('out/val_fast.json'))
print('idea CER', idea['cer'], 'fast CER', fast['cer'], 'ratio', round(fast['cer'] / idea['cer'], 3))
NFE = 16
if fast['cer'] > 1.2 * idea['cer']:
    print('nfe 16 breaks the 1.2x CER limit -> trying nfe 24 on GPU 0')
    sh(f'CUDA_VISIBLE_DEVICES=0 python baseline/run_baseline.py --items data/val_items.csv --out out/val_fast24 {T3} --set nfe_step=24', tail=3)
    sh('python metrics.py out/val_fast24 --json out/val_fast24.json --csv out/val_fast24.csv', tail=12)
    NFE = 24
print('fast system uses nfe_step =', NFE)

In [ ]:
src = 'out/val_fast' if NFE == 16 else 'out/val_fast24'
sh(f'CUDA_VISIBLE_DEVICES=0 python baseline/run_baseline.py --items data/eval_items.csv --out out/t4_fast {T3} --set nfe_step={NFE}', tail=3)
for a, b in [('out/val_idea', 'predictions/val_idea'), ('out/t3_idea', 'predictions/t3_idea'), ('out/t4_fast', 'predictions/t4_fast')]:
    if os.path.isdir(a):
        shutil.copytree(a, b, dirs_exist_ok=True)
open('out/fast_choice.txt', 'w').write(f'{NFE} {src}')
import zipfile
z = zipfile.ZipFile('/kaggle/working/run2_outputs.zip', 'w', zipfile.ZIP_DEFLATED)
for base in ['predictions', 'out']:
    for root, _, files in os.walk(base):
        for f in files:
            z.write(os.path.join(root, f))
z.close()
print('wrote run2_outputs.zip', os.path.getsize('/kaggle/working/run2_outputs.zip') // 1024, 'KB')